In [15]:
from netCDF4 import Dataset
import re
import plot_scripts as plot_scripts
import os
import pandas as pd
import numpy as np
from astral.sun import sun
from astral import LocationInfo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import time
import warnings
from datetime import time
import xarray as xr
from scipy.spatial import cKDTree
import drought_utils as du


# Suppress specific warnings
warnings.filterwarnings("ignore", message="WARNING: valid_range not used since it")

In [16]:
# # Function to assign seasons based on hemisphere
def assign_season(month, latitude):
    if latitude >= 0:  # Northern Hemisphere
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [6, 7, 8]:
            return 'Summer'
        elif month in [3, 4, 5]:
            return 'Spring'
        elif month in [9, 10, 11]:
            return 'Fall'
        else:
            return 'Other'
    else:  # Southern Hemisphere
        if month in [6, 7, 8]:
            return 'Winter'
        elif month in [12, 1, 2]:
            return 'Summer'
        elif month in [3, 4, 5]:
            return 'Fall'
        elif month in [9, 10, 11]:
            return 'Spring'
        else:
            return 'Other'
        
nc_path = "data/Support/koppen_geiger_0p1.nc"
ds = xr.open_dataset(nc_path)
lat = ds['lat'].values
lon = ds['lon'].values
kg_class = ds['kg_class'].values
kg_confidence = ds['kg_confidence'].values

lat_grid, lon_grid = np.meshgrid(lat, lon, indexing='ij')
points = np.column_stack((lat_grid.ravel(), lon_grid.ravel()))
tree = cKDTree(points)

drought_folder = "data/Support/Drought"
spei_index = du.load_drought_index(drought_folder, prefix="SPEI1", variable="SPEI1")
spi_index = du.load_drought_index(drought_folder, prefix="SPI1", variable="SPI1")

def extract_climate_for_site(lat, lon):
    # Find the nearest grid point
    distance, idx = tree.query((lat, lon))
    
    # Return the climate class and confidence for the nearest grid point
    return kg_class.ravel()[idx], kg_confidence.ravel()[idx]    

def extract_drought_for_site(lat, lon, timestamp):
    return du.extract_drought_index(spei_index, lat, lon, timestamp)

def extract_spi_for_site(lat, lon, timestamp):
    return du.extract_drought_index(spi_index, lat, lon, timestamp)

koppen_labels = {
    1: "Af",   2: "Am",   3: "Aw",
    4: "BWh",  5: "BWk",  6: "BSh",  7: "BSk",
    8: "Csa",  9: "Csb", 10: "Csc",
    11: "Cwa", 12: "Cwb", 13: "Cwc",
    14: "Cfa", 15: "Cfb", 16: "Cfc",
    17: "Dsa", 18: "Dsb", 19: "Dsc",
    20: "Dsd", 21: "Dwa", 22: "Dwb",
    23: "Dwc", 24: "Dwd", 25: "Dfa",
    26: "Dfb", 27: "Dfc", 28: "Dfd",
    29: "ET",  30: "EF"
}

igbp_classes = {
    0: 'Unknown',
    1: 'ENF',
    2: 'EBF',
    3: 'DNF',
    4: 'DBF',
    5: 'MF',
    6: 'CSH',
    7: 'OSH',
    8: 'WSA',
    9: 'SAV',
    10: 'GRA',
    11: 'WET',
    12: 'CRO',
    13: 'URB',
    14: 'CVM',
    15: 'SNO',
    16: 'BSV',
    17: 'WAT'
}

# Empirical global SIF->GPP conversion factor, derived from coincident
# FLUXNET GPP vs ECOCO3 SIF at matched tower/pixel locations (see
# sif_gpp_conversion.py). Replaces the earlier per-IGBP-class literature
# lookup table -- fit quality was comparable (R^2 ~0.21-0.26 either way)
# and the coincident sample was too small to support reliable
# per-vegetation factors for most classes.
SIF_TO_GPP_FACTOR = 14.446

In [17]:
print(ds_drought['time'])

<xarray.DataArray 'time' (time: 910)> Size: 7kB
array(['1950-01-01T00:00:00.000000000', '1950-02-01T00:00:00.000000000',
       '1950-03-01T00:00:00.000000000', ..., '2025-08-01T00:00:00.000000000',
       '2025-09-01T00:00:00.000000000', '2025-10-01T00:00:00.000000000'],
      shape=(910,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2025-10-01
Attributes:
    long_name:  time


In [18]:
# data_folder = 'ECOCO3Test Files20241113031233/'
# data_folder = 'temp_test/'
data_folder = 'data/'
folder_info = 'ECOCO3_V2/'

# Step 1: Gather all .nc4 file paths
nc4_files = []
for root, dirs, files in os.walk(data_folder + folder_info):
    for fname in files:
        if fname.endswith('.nc4'):
            nc4_files.append(os.path.join(root, fname))

# Step 2: Get the total number of files
total_files = len(nc4_files)
print(f"Total number of .nc4 files found: {total_files}")

# Initialize a list to store the extracted data
extracted_data = []
extracted_data_daily = []

def to_numpy(arr):
    if np.issubdtype(arr.dtype, np.integer):
        arr = arr.astype(float)
    if np.ma.isMaskedArray(arr):
        arr = arr.filled(np.nan)
    return np.asarray(arr).ravel()


# Regex pattern to match the datetime in the filename
datetime_pattern = r'(\d{4})(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})'
count = 0
# Walk through all subdirectories in the 'unzipped' folder
for root, dirs, files in os.walk(data_folder+folder_info):
    # Loop through each .nc4 file found
    for fname in files:
        #if fname.endswith('.nc4') and ('ecoco3_eco' in fname or 'ecoco3_sif' in fname):
        if fname.endswith('.nc4'):

            # Open the NetCDF file
            print(f"Processing file {count+1}/{total_files}: {fname}")
            
            nc = Dataset(os.path.join(root, fname))
            
            site_lat = float(nc['OCO3_Sequence_Site'].variables['oco_site_centroid_location'][1])
            site_lon = float(nc['OCO3_Sequence_Site'].variables['oco_site_centroid_location'][0])
            parts = fname.split('_')
            
            # Extract the identifier from the filename and look up the full name
            site_name = parts[1]

            # Search for the datetime pattern in the filename
            match = re.search(datetime_pattern, fname)

            if match:
                datetime_str = f"{match.group(1)}/{match.group(2)}/{match.group(3)} {match.group(4)}:{match.group(5)}:{match.group(6)}"
            
            datetime_parsed = pd.to_datetime(datetime_str, errors='coerce')
            if pd.isna(datetime_parsed):
                print(f"Skipping: {site_name} due to invalid datetime in filename.")
                nc.close()
                continue

            utc_offset=site_lon/15
            #datetime_solar = datetime_parsed + pd.to_timedelta(int(utc_offset), unit='h')
            datetime_solar = datetime_parsed + pd.to_timedelta(utc_offset, unit='h')
            count += 1

            if not (datetime_solar.time() >= time(5, 0) and datetime_solar.time() <= time(21, 0)):
                print(f"Skipping: {site_name} at {datetime_solar} (Sun is down)")
                nc.close()
                continue
    
            # Access variables
            pixel_lat = to_numpy(nc['Geolocation'].variables['latitude'][:])
            pixel_lon = to_numpy(nc['Geolocation'].variables['longitude'][:])
            eco_et = to_numpy(nc['Science'].variables['eco_ptjplsm_inst'][:])
            oco_sif = to_numpy(nc['Science'].variables['oco_sif_740nm'][:])
            eco_lst = to_numpy(nc['Science'].variables['eco_lst'][:])
            igbp_modis = to_numpy(nc['Science'].variables['igbp17_type_modis'][:,:,0])
            igbp_viirs = to_numpy(nc['Science'].variables['igbp17_type_viirs'][:,:,0])
            phase_angle = to_numpy(nc['Science'].variables['oco_sif_phase_angle'][:])
            oco_sif_uncert = to_numpy(nc['Science'].variables['oco_sif_740nm_uncert'][:])

            wue = oco_sif / eco_et

            eco_et_daily = to_numpy(nc['Science'].variables['eco_et_daily'][:])
            #eco_et_daily *= 0.03521 / 2  # Convert to mm/day because previously in W/m2 and now in mm/day --- IGNORE ---

            oco_sif_daily = to_numpy(nc['Science'].variables['oco_sif_740nm_daily'][:])
            wue_eco_daily = to_numpy(nc['Science'].variables['eco_wue'][:])
            wue_daily = oco_sif_daily / eco_et_daily
            
            # Close the NetCDF file
            nc.close()

            # Plot the data and save the figure
            os.makedirs('figures/'+folder_info, exist_ok=True)
            out_path = os.path.join('figures/'+folder_info, f'{fname}.png')
            #plot_scripts.plot_maps(lat, lon, eco_et, oco_sif, wue, site_name, datetime_solar, out_path=out_path)
            #plot_scripts.plot_maps_daily(lat, lon, eco_et, oco_sif, wue, wue_eco_daily, site_name, datetime_str, out_path=out_path)

            df = pd.DataFrame({
                'WUE': wue,
                'IGBP_MODIS': igbp_modis,
                'IGBP_VIIRS': igbp_viirs,
                'SIF': oco_sif,
                'ET': eco_et,
                'LST': eco_lst,
                'Phase_Angle': phase_angle,
                'SIF_Uncertainty': oco_sif_uncert, 
            })
            df['IGBP'] = df['IGBP_MODIS']
            df = df[df['IGBP_MODIS']==df['IGBP_VIIRS']]  # Filter out rows where IGBP_MODIS is NaN

            df = df[df['ET'] > 5]  # Filter out rows where ET is zero or negative to avoid division issues
            df = df.groupby(['IGBP_MODIS', 'IGBP_VIIRS']).mean().reset_index()

            df['WUE_gCkgH20'] = df['WUE'] * SIF_TO_GPP_FACTOR
            df['WUE_gCkgH20'] *= 2/0.03521  # Convert to mm/day using quick and dirty conversion factor from ET paper flipped around
            df['GPP'] = df['SIF'] * SIF_TO_GPP_FACTOR

            if df.empty:
                print(f"Skipping: {site_name} at {datetime_solar} (No valid data after filtering)")
                continue
            
            df['IGBP_class_MODIS'] = df['IGBP_MODIS'].map(igbp_classes)
            df['IGBP_class_VIIRS'] = df['IGBP_VIIRS'].map(igbp_classes)

            df['SiteName'] = site_name
            df['Lat'] = site_lat
            df['Lon'] = site_lon
            df['Timestamp'] = datetime_parsed
            df['LocalTime'] = pd.to_datetime(datetime_solar).round('30min')
            df['TOD'] = df['LocalTime'].dt.hour.map(lambda x: 'Morning' if x in [7, 8, 9, 10] else 'Midday' if x in [11, 12, 13, 14] else 'Afternoon' if x in [15, 16, 17, 18] else 'Other')
            df['Season'] = df.apply(lambda row: assign_season(int(row['LocalTime'].month), float(row['Lat'])), axis=1)
            df[['kg_class', 'kg_confidence']] = df.apply(
                lambda row: extract_climate_for_site(row['Lat'], row['Lon']),
                axis=1, result_type='expand'
            )
            df['kg_label'] = df['kg_class'].map(koppen_labels)
            
            df['SPEI']= df.apply(lambda row: extract_drought_for_site(row['Lat'], row['Lon'], row['Timestamp']), axis=1)
            df['SPI'] = df.apply(lambda row: extract_spi_for_site(row['Lat'], row['Lon'], row['Timestamp']), axis=1)

            df = df.dropna()
     
            extracted_data.append(df)

            df_daily = pd.DataFrame({
                'WUE': wue_daily,
                'WUE_ECO': wue_eco_daily,
                'IGBP_MODIS': igbp_modis,
                'IGBP_VIIRS': igbp_viirs,
                'SIF': oco_sif_daily,
                'ET': eco_et_daily,
                'Phase_Angle': phase_angle,
                'SIF_Uncertainty': oco_sif_uncert
            })
            df_daily['IGBP'] = df_daily['IGBP_MODIS']
            df_daily = df_daily[df_daily['IGBP_MODIS']==df_daily['IGBP_VIIRS']]  # Filter out rows where IGBP_MODIS is NaN

            df_daily = df_daily[df_daily['ET'] > 5 * 0.03521 / 2]  # Filter out rows where ET is zero or negative to avoid division issues
            df_daily = df_daily.groupby(['IGBP_MODIS', 'IGBP_VIIRS']).mean().reset_index()
            df_daily['WUE_gCkgH20'] = df_daily['WUE'] * SIF_TO_GPP_FACTOR
            df_daily['GPP'] = df_daily['SIF'] * SIF_TO_GPP_FACTOR

            if df_daily.empty:
                print(f"Skipping: {site_name} at {datetime_solar} (No valid daily data after filtering)")
                continue

            df_daily['IGBP_class_MODIS'] = df_daily['IGBP_MODIS'].map(igbp_classes)
            df_daily['IGBP_class_VIIRS'] = df_daily['IGBP_VIIRS'].map(igbp_classes)

            df_daily['SiteName'] = site_name
            df_daily['Lat'] = site_lat
            df_daily['Lon'] = site_lon
            df_daily['Timestamp'] = datetime_parsed
            df_daily['LocalTime'] = pd.to_datetime(datetime_solar).round('30min')
            df_daily['TOD'] = df_daily['LocalTime'].dt.hour.map(lambda x: 'Morning' if x in [7, 8, 9, 10] else 'Midday' if x in [11, 12, 13, 14] else 'Afternoon' if x in [15, 16, 17, 18] else 'Other')
            df_daily['Season'] = df_daily.apply(lambda row: assign_season(int(row['LocalTime'].month), float(row['Lat'])), axis=1)
            df_daily[['kg_class', 'kg_confidence']] = df_daily.apply(
                lambda row: extract_climate_for_site(row['Lat'], row['Lon']),
                axis=1, result_type='expand'
            )
            df_daily['kg_label'] = df_daily['kg_class'].map(koppen_labels)
            df_daily['SPEI']= df_daily.apply(lambda row: extract_drought_for_site(row['Lat'], row['Lon'], row['Timestamp']), axis=1)
            df_daily['SPI'] = df_daily.apply(lambda row: extract_spi_for_site(row['Lat'], row['Lon'], row['Timestamp']), axis=1)
            df_daily = df_daily.dropna()

            extracted_data_daily.append(df_daily)

# Convert the list to a DataFrame
df_wue_daily = pd.concat(extracted_data_daily, ignore_index=True)
# Convert the list to a DataFrame
df_wue = pd.concat(extracted_data, ignore_index=True)

Total number of .nc4 files found: 11681
Processing file 1/11681: ecoco3_fos115_20220303014429_v200_20260506t040525z.nc4
Processing file 2/11681: ecoco3_vol040_20220303173059_v200_20260506t191557z.nc4
Processing file 3/11681: ecoco3_fos135_20220303153639_v200_20260506t085309z.nc4
Processing file 4/11681: ecoco3_fos174_20220303044657_v200_20260506t214705z.nc4
Processing file 5/11681: ecoco3_fos121_20220304144809_v200_20260506t062457z.nc4
Processing file 6/11681: ecoco3_cal009_20220304083749_v200_20260505t112635z.nc4
Processing file 7/11681: ecoco3_fos084_20220304164220_v200_20260506t100603z.nc4
Processing file 8/11681: ecoco3_coc101_20220304102729_v200_20260505t133913z.nc4
Processing file 9/11681: ecoco3_fos040_20220304005209_v200_20260505t192724z.nc4
Processing file 10/11681: ecoco3_fos157_20220304040019_v200_20260506t151724z.nc4
Processing file 11/11681: ecoco3_fos161_20220304052900_v200_20260506t170730z.nc4
Processing file 12/11681: ecoco3_vol066_20220304145719_v200_20260506t214844z.n

/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily


Processing file 14/11681: ecoco3_fos132_20220304005428_v200_20260506t082623z.nc4
Processing file 15/11681: ecoco3_fos043_20220305000439_v200_20260505t203429z.nc4
Processing file 16/11681: ecoco3_vol093_20220305173219_v200_20260507t023156z.nc4
Processing file 17/11681: ecoco3_fos160_20220305044328_v200_20260506t165541z.nc4
Processing file 18/11681: ecoco3_fos033_20220302131229_v200_20260505t161600z.nc4
Processing file 19/11681: ecoco3_vol008_20220302181909_v200_20260506t124710z.nc4
Processing file 20/11681: ecoco3_tcc102_20220302161959_v200_20260505t220937z.nc4
Processing file 21/11681: ecoco3_tcc115_20220320012458_v200_20260506t022746z.nc4


/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily


Processing file 22/11681: ecoco3_vol012_20220320215149_v200_20260506t144017z.nc4
Processing file 23/11681: ecoco3_fos036_20220320232759_v200_20260505t174615z.nc4
Processing file 24/11681: ecoco3_fos202_20220320043858_v200_20260506t160925z.nc4
Processing file 25/11681: ecoco3_fos223_20220320122220_v200_20260506t195321z.nc4
Processing file 26/11681: ecoco3_fos101_20220320201137_v200_20260505t234223z.nc4
Processing file 27/11681: ecoco3_vol011_20220320154129_v200_20260506t142917z.nc4
Processing file 28/11681: ecoco3_fos084_20220320183128_v200_20260506t100744z.nc4
Processing file 29/11681: ecoco3_tcc135_20220318043559_v200_20260506t090945z.nc4
Processing file 30/11681: ecoco3_fos135_20220327210709_v200_20260506t085437z.nc4
Processing file 31/11681: ecoco3_fos086_20220327095719_v200_20260506t115900z.nc4
Processing file 32/11681: ecoco3_coc102_20220327100048_v200_20260505t145956z.nc4
Processing file 33/11681: ecoco3_vol076_20220327161229_v200_20260506t222328z.nc4
Processing file 34/11681: ec

...

Processing file 11660/11681: ecoco3_fos191_20201213193230_v200_20260507t034206z.nc4
Processing file 11661/11681: ecoco3_tcc100_20201213004820_v200_20260505t215102z.nc4
Processing file 11662/11681: ecoco3_tcc124_20201213162159_v200_20260506t053610z.nc4
Processing file 11663/11681: ecoco3_sif021_20201214153509_v200_20260505t080127z.nc4
Processing file 11664/11681: ecoco3_tcc128_20201214000448_v200_20260506t070413z.nc4
Processing file 11665/11681: ecoco3_fos129_20201214013558_v200_20260506t081634z.nc4
Processing file 11666/11681: ecoco3_fos075_20201214092329_v200_20260506t070334z.nc4
Processing file 11667/11681: ecoco3_eco026_20201214092529_v200_20260505t133546z.nc4


/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily


Processing file 11668/11681: ecoco3_fos029_20201222080809_v200_20260505t140629z.nc4
Processing file 11669/11681: ecoco3_fos142_20201222203409_v200_20260506t112251z.nc4
Processing file 11670/11681: ecoco3_vol045_20201222032348_v200_20260506t195806z.nc4
Processing file 11671/11681: ecoco3_fos026_20201222171959_v200_20260505t133055z.nc4
Processing file 11672/11681: ecoco3_fos114_20201222093149_v200_20260506t032608z.nc4
Processing file 11673/11681: ecoco3_fos009_20201222032649_v200_20260505t083946z.nc4
Processing file 11674/11681: ecoco3_fos005_20201222202909_v200_20260505t060934z.nc4
Processing file 11675/11681: ecoco3_fos057_20201222185339_v200_20260506t011638z.nc4


/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_daily / eco_et_daily
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:84: RuntimeWarning: divide by zero encountered in divide
  wue = oco_sif / eco_et
/var/folders/w5/9r5q7sgj0cs5jn_vq73jqdq00000gn/T/ipykernel_91053/3953021295.py:91: RuntimeWarning: divide by zero encountered in divide
  wue_daily = oco_sif_d

Processing file 11676/11681: ecoco3_fos038_20201225041627_v200_20260505t181741z.nc4
Processing file 11677/11681: ecoco3_fos149_20201225180602_v200_20260506t130032z.nc4
Processing file 11678/11681: ecoco3_fos064_20201225163142_v200_20260506t041627z.nc4
Processing file 11679/11681: ecoco3_fos044_20201225023658_v200_20260505t203916z.nc4
Processing file 11680/11681: ecoco3_tmx005_20201225180842_v200_20260505t162604z.nc4
Processing file 11681/11681: ecoco3_eco041_20201225043759_v200_20260505t161919z.nc4


In [19]:
df_wue_daily.to_csv(data_folder+'ECOCO3_cleaned/'+folder_info.rstrip('/')+"_df_wue_daily_fullset.csv", index=False)
df_wue.to_csv(data_folder+'ECOCO3_cleaned/'+folder_info.rstrip('/')+"_df_wue_fullset.csv", index=False)